In [1]:
from pathlib import Path

import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

INPUT_PATH = PROCESSED_DATA_DIR / "movies_features.csv"

TFIDF_PATH = MODELS_DIR / "tfidf_vectorizer.joblib"
SIMILARITY_PATH = MODELS_DIR / "similarity_matrix.joblib"
MOVIES_PATH = MODELS_DIR / "movies_metadata.joblib"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
movies_df = pd.read_csv(INPUT_PATH)

display(movies_df.head())

,id,title,tags
0,19995,Avatar,action adventure fantasy science fiction cultu...
1,285,Pirates of the Caribbean: At World's End,adventure fantasy action ocean drug abuse exot...
2,206647,Spectre,action adventure crime spy based on novel secr...
3,49026,The Dark Knight Rises,action crime drama thriller dc comics crime fi...
4,49529,John Carter,action adventure science fiction based on nove...


In [4]:
print("Dataset shape:", movies_df.shape)

print("\nColumns:")
print(movies_df.columns.tolist())

print("\nMissing values:")
print(movies_df.isnull().sum())

print("\nDuplicate movie IDs:", movies_df["id"].duplicated().sum())

print("\nEmpty tags:", (movies_df["tags"].fillna("").str.strip() == "").sum())

Dataset shape: (4803, 3)

Columns:
['id', 'title', 'tags']

Missing values:
id       0
title    0
tags     0
dtype: int64

Duplicate movie IDs: 0

Empty tags: 0


In [6]:
required_columns = ["id", "title", "tags"]

missing_columns = [
    column for column in required_columns if column not in movies_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

In [14]:
movies_df["title"] = movies_df["title"].fillna("").str.strip()
movies_df["tags"] = movies_df["tags"].fillna("").astype(str).str.strip()

movies_df = movies_df[(movies_df["title"] != "") & (movies_df["tags"] != "")].copy()

movies_df = movies_df.reset_index(drop=True)

print("Dataset shape after validation:", movies_df.shape)

Dataset shape after validation: (4803, 3)


In [15]:
tfidf = TfidfVectorizer(max_features=5000, stop_words="english")

tfidf_matrix = tfidf.fit_transform(movies_df["tags"])

In [16]:
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of movies:", tfidf_matrix.shape[0])
print("Number of features:", tfidf_matrix.shape[1])
print("Matrix type:", type(tfidf_matrix))

TF-IDF matrix shape: (4803, 5000)
Number of movies: 4803
Number of features: 5000
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>


In [17]:
feature_names = tfidf.get_feature_names_out()

print("First 30 TF-IDF vocabulary terms:")
print(feature_names[:30])

First 30 TF-IDF vocabulary terms:
['000' '007' '10' '100' '11' '12' '13' '14' '15' '16' '17' '17th' '18'
 '18th' '19' '1930s' '1940s' '1950s' '1960s' '1970s' '1980s' '19th' '20'
 '200' '20th' '21st' '24' '25' '30' '300']


In [18]:
similarity_matrix = cosine_similarity(tfidf_matrix, dense_output=False)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (4803, 4803)


In [19]:
def recommend_movies(movie_title: str, top_n: int = 5) -> pd.DataFrame:
    """
    Recommend movies based on content similarity.

    Parameters
    ----------
    movie_title : str
        Title of the movie used as the recommendation reference.
    top_n : int, default=5
        Number of recommendations to return.

    Returns
    -------
    pd.DataFrame
        Recommended movie titles and their similarity scores.
    """
    if not isinstance(movie_title, str) or not movie_title.strip():
        raise ValueError("movie_title must be a non-empty string.")

    if not isinstance(top_n, int) or isinstance(top_n, bool) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")

    matches = movies_df.index[
        movies_df["title"].str.casefold() == movie_title.strip().casefold()
    ].tolist()

    if not matches:
        raise ValueError(f"Movie '{movie_title}' was not found.")

    movie_index = matches[0]

    similarity_scores = similarity_matrix.getrow(movie_index).toarray().ravel()

    similar_indices = similarity_scores.argsort()[::-1]

    similar_indices = [index for index in similar_indices if index != movie_index][
        :top_n
    ]

    recommendations = movies_df.iloc[similar_indices][["id", "title"]].copy()

    recommendations["similarity_score"] = similarity_scores[similar_indices]

    return recommendations.reset_index(drop=True)

In [20]:
recommend_movies("Avatar", top_n=5)

,id,title,similarity_score
0,679,Aliens,0.429577
1,348,Alien,0.363222
2,698,Moonraker,0.349246
3,8077,Alien³,0.347270
4,811,Silent Running,0.315075


In [21]:
recommend_movies("The Dark Knight", top_n=5)

,id,title,similarity_score
0,49026,The Dark Knight Rises,0.510067
1,272,Batman Begins,0.438464
2,364,Batman Returns,0.388453
3,414,Batman Forever,0.380468
4,209112,Batman v Superman: Dawn of Justice,0.317970


In [22]:
recommend_movies("Iron Man", top_n=5)

,id,title,similarity_score
0,10138,Iron Man 2,0.565749
1,68721,Iron Man 3,0.542642
2,99861,Avengers: Age of Ultron,0.437284
3,271110,Captain America: Civil War,0.357591
4,1724,The Incredible Hulk,0.343927


In [23]:
recommendations = recommend_movies("Avatar", top_n=10)

display(recommendations.style.format({"similarity_score": "{:.4f}"}))

,id,title,similarity_score
0,679,Aliens,0.4296
1,348,Alien,0.3632
2,698,Moonraker,0.3492
3,8077,Alien³,0.3473
4,811,Silent Running,0.3151
5,957,Spaceballs,0.3069
6,2067,Mission to Mars,0.2845
7,2157,Lost in Space,0.2839
8,11954,Lifeforce,0.2756
9,81796,Lockout,0.2746


In [24]:
movie_metadata = movies_df[["id", "title"]].copy()

joblib.dump(tfidf, TFIDF_PATH)
joblib.dump(similarity_matrix, SIMILARITY_PATH)
joblib.dump(movie_metadata, MOVIES_PATH)

print("Model artifacts saved successfully.")
print("TF-IDF:", TFIDF_PATH)
print("Similarity matrix:", SIMILARITY_PATH)
print("Movie metadata:", MOVIES_PATH)

Model artifacts saved successfully.
TF-IDF: d:\Learning-Materials\Projects\ML-Projects\Content-Based-Movie-Recommendation-System\models\tfidf_vectorizer.joblib
Similarity matrix: d:\Learning-Materials\Projects\ML-Projects\Content-Based-Movie-Recommendation-System\models\similarity_matrix.joblib
Movie metadata: d:\Learning-Materials\Projects\ML-Projects\Content-Based-Movie-Recommendation-System\models\movies_metadata.joblib


In [25]:
loaded_tfidf = joblib.load(TFIDF_PATH)
loaded_similarity = joblib.load(SIMILARITY_PATH)
loaded_movies = joblib.load(MOVIES_PATH)

print("Vectorizer loaded:", isinstance(loaded_tfidf, TfidfVectorizer))
print("Similarity matrix shape:", loaded_similarity.shape)
print("Movie metadata shape:", loaded_movies.shape)

Vectorizer loaded: True
Similarity matrix shape: (4803, 4803)
Movie metadata shape: (4803, 2)


In [26]:
FEATURES_PATH = MODELS_DIR / "movie_features.joblib"

movie_features = movies_df[["id", "title", "tags"]].copy()

joblib.dump(movie_features, FEATURES_PATH)

['d:\\Learning-Materials\\Projects\\ML-Projects\\Content-Based-Movie-Recommendation-System\\models\\movie_features.joblib']

## Conclusion

In this notebook, we implemented the first version of our
Content-Based Movie Recommendation System.

### Work completed

- Loaded the processed movie feature dataset.
- Validated movie records and textual features.
- Converted movie tags into TF-IDF vectors.
- Calculated cosine similarity between movies.
- Implemented a reusable top-N recommendation function.
- Tested recommendations using multiple movie titles.
- Saved the model artifacts for future use.

### Next step

In the next notebook, we will evaluate the recommendation system,
inspect recommendation quality, and identify opportunities for
improvement.

In [27]:
movie_metadata = movies_df[["id", "title"]].copy()

joblib.dump(tfidf, MODELS_DIR / "tfidf_vectorizer.joblib")
joblib.dump(tfidf_matrix, MODELS_DIR / "tfidf_matrix.joblib")
joblib.dump(movie_metadata, MODELS_DIR / "movies_metadata.joblib")

print("Model artifacts saved successfully.")

Model artifacts saved successfully.
